# 🚀 生成式AI應用開發｜第03週
## OpenAI API 入門：從 API Key 到第一個 Python AI 函式

> **執行環境：Google Colab** ｜ **本週開始使用 OpenAI API**

本週目標是把第 2 週練過的 Python 技能接到真實 LLM API。你會完成：

1. 安全讀取 `OPENAI_API_KEY`
2. 建立 OpenAI client
3. 用 Responses API 送出第一個請求
4. 讀取文字輸出、response id、<font color="#1a73e8"><b>usage</b></font>
5. 封裝成可重複使用的 Python 函式
6. 加入基本錯誤處理

官方文件入口：
- Text generation: https://platform.openai.com/docs/guides/text
- Responses API reference: https://platform.openai.com/docs/api-reference/responses
- Quickstart: https://platform.openai.com/docs/quickstart

<div style="border-left:4px solid #1a73e8; padding:10px 12px; background:#eef4ff; margin:10px 0;">
  <font color="#1a73e8"><b>本週任務</b></font><br>
  完成第一個 API 呼叫，並把呼叫流程封裝成可重複使用的 Python 函式。
</div>


> **學生版**：本檔保留課堂練習的 TODO 骨架，請依照註解完成程式。
> 需要執行 OpenAI API 的 cells 會產生費用，請依老師指示執行。

## 🎯 本週學習目標

| # | 能力 | 對應後續課程 |
|---|------|-------------|
| 1 | 安裝與匯入 OpenAI SDK | 所有 API 實作基礎 |
| 2 | 使用 Colab Secrets 讀取 API key | 安全管理金鑰 |
| 3 | 呼叫 `client.responses.create()` | 文字生成、Prompt Engineering |
| 4 | 使用 `instructions` 與 `input` | system prompt / user prompt |
| 5 | 解析 `output_text` 與 <font color="#1a73e8"><b>usage</b></font> | 顯示回答、估算成本、除錯 |
| 6 | 封裝 API 呼叫函式 | Week 4-7 的 App 開發 |
| 7 | 加入錯誤處理 | 網路、金鑰、額度與格式問題 |


---
## 🧰 第一節：安裝套件

Colab 每次重新開啟 runtime 後，都需要重新安裝套件。

In [ ]:
# 安裝 OpenAI Python SDK
!pip install openai --quiet

print("OpenAI SDK 安裝完成")

In [ ]:
# 匯入本週使用的模組
import os
import json
from openai import OpenAI

print("模組載入完成")

---
## 🔐 第二節：設定 API Key

請在 Colab 左側的 **Secrets** 面板新增：

- Name: `OPENAI_API_KEY`
- Value: 你的 OpenAI API key
- 開啟 Notebook access

<font color="#b00020"><b>不要把真正的 API key</b></font> 寫在 notebook 裡，也不要截圖或貼到作業中。

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>安全提醒</b></font><br>
  <font color="#b00020"><b>API key 是機密資訊</b></font>。請使用 Colab Secrets 或環境變數，不要直接寫在 notebook、作業或截圖中。
</div>


In [ ]:
# 從 Colab Secrets 讀取 API key，並轉成環境變數
try:
    from google.colab import userdata
    api_key = userdata.get("OPENAI_API_KEY")
    if api_key:
        os.environ["OPENAI_API_KEY"] = api_key
        print(f"已讀取 API key（前 8 碼）：{api_key[:8]}...")
    else:
        print("Secrets 中找不到 OPENAI_API_KEY，請先完成設定")
except Exception as e:
    print(f"目前可能不是在 Colab 執行：{e}")
    print("若在本機執行，請先設定環境變數 OPENAI_API_KEY")

print("目前讀取狀態：", "已設定" if os.getenv("OPENAI_API_KEY") else "未設定")

### 💻 本機環境變數設定方式

```powershell
# Windows PowerShell：只在目前視窗有效
$env:OPENAI_API_KEY="sk-..."

# Windows：寫入使用者環境變數，設定後請重新開啟終端機
setx OPENAI_API_KEY "sk-..."
```

```bash
# macOS / Linux：只在目前 shell session 有效
export OPENAI_API_KEY="sk-..."
```


---
## 🔌 第三節：建立 OpenAI Client

`OpenAI()` 會自動從環境變數 `OPENAI_API_KEY` 讀取金鑰。

In [ ]:
# 建立 OpenAI client
client = OpenAI()

# 課堂預設使用較低成本、較適合入門練習的 mini 模型。
# 若老師課前確認其他模型更適合，可在 Colab 或本機設定 OPENAI_MODEL 覆蓋。
MODEL = os.getenv("OPENAI_MODEL", "gpt-5.4-mini")

print(f"Client 建立完成，使用模型：{MODEL}")

---
## ✨ 第四節：第一個 Responses API 呼叫

官方文件目前的文字生成範例使用 `client.responses.create()`，並以 `response.output_text` 取出文字輸出。

In [ ]:
# 第一個 API 呼叫：請確認 API key 已設定後再執行
response = client.responses.create(
    model=MODEL,
    input="請用繁體中文，用三句話解釋什麼是生成式 AI。"
)

print(response.output_text)

### 🔍 4-1 檢查 response 物件

實務開發時，不只要看回答文字，也要知道：

- `id`：這次 API 呼叫的識別碼，方便除錯
- `model`：實際使用的模型
- `<font color="#1a73e8"><b>usage</b></font>`：token 使用量


In [ ]:
print("response id:", response.id)
print("model:", response.model)
print("output_text:")
print(response.output_text)

print()
print("usage:")
print(response.usage)

---
## 🧭 第五節：加入 instructions

`instructions` 是給模型的「任務規則」或「角色設定」。它通常用來放比較穩定、不會每次都改變的要求，例如：

- 你要扮演什麼角色
- 回答要使用什麼語言
- 回答要簡短或詳細
- 是否要使用條列、表格或固定格式
- 有哪些安全或教學上的限制

在 Responses API 中，可以把「模型應該怎麼回答」放在 `instructions`，把「使用者這次問什麼」放在 `input`。這種寫法很適合單次任務，也適合初學者先理解 API 呼叫的基本結構。

概念上可以把它想成：

```text
instructions = 老師先交代助教的工作規則
input        = 學生這一次提出的問題
```

下一格範例中，`instructions` 負責設定「資管系 Python 助教」與回答風格，`input` 則是本次真正要回答的問題。


In [ ]:
response = client.responses.create(
    model=MODEL,
    instructions="你是一位資管系 Python 助教。回答要精簡、具體，並使用繁體中文。",
    input="請解釋 API key 為什麼不能直接寫在程式碼裡。"
)

print(response.output_text)

---
## 💬 第六節：用 role/content 格式建立輸入

除了 `instructions + input`，Responses API 的 `input` 也可以使用由 `role` 和 `content` 組成的訊息列表。這種格式更接近聊天紀錄，也和第 2 週練過的 `messages` list/dict 結構直接銜接。

常見角色：

- `developer`：開發者或系統層級指令，用來規範模型行為
- `user`：使用者輸入，代表這一輪真正提出的問題
- `assistant`：模型先前的回答，常用於保存多輪對話脈絡

### `instructions` 與 role/content 的差異

| 寫法 | 重點 | 適合情境 |
|---|---|---|
| `instructions + input` | 把規則和本次問題分開，結構簡潔 | 單次問答、簡單函式、入門練習 |
| role/content list | 把每一則訊息都明確標示角色，可累積上下文 | 多輪對話、聊天機器人、Streamlit App、歷史紀錄 |

兩種寫法都可以設定模型行為，但思考方式不同：

```text
instructions + input：這次任務的規則 + 這次的問題
role/content list：一串有角色標籤的對話紀錄
```

在小型練習中，`instructions + input` 通常比較清楚；在要做聊天介面、保存歷史訊息、或讓使用者連續追問時，role/content list 比較容易擴充。


In [ ]:
messages = [
    {
        "role": "developer",
        "content": "你是一位教學助理，回答要適合初學 Python 的大學生。"
    },
    {
        "role": "user",
        "content": "請用生活化例子說明什麼是 API。"
    }
]

response = client.responses.create(
    model=MODEL,
    input=messages
)

print(response.output_text)

### 🧠 6-1 什麼時候選哪一種？

實務上可以用以下規則判斷：

1. 如果只是「輸入一段文字，得到一次回答」，優先使用 `instructions + input`。
2. 如果需要保存使用者前面問過什麼、AI 前面回答過什麼，使用 role/content list。
3. 如果要做 Streamlit 聊天機器人，通常會把每一輪對話存成 list，再傳給 API。
4. 如果規則很固定，例如「永遠使用繁體中文、用資管系學生能懂的例子」，可以放在 `instructions` 或 `developer` 訊息中。

本課程後續會先用 `instructions + input` 建立簡單函式，再逐步改成 role/content list，最後銜接 Streamlit 的對話狀態管理。


### 🔁 6-2 Responses API 的多輪對話：previous_response_id

OpenAI Responses API 也可以延續上一輪對話。做法是保留第一次回應的 `response.id`，下一次呼叫時傳入 `previous_response_id`。

這和 Gemini 的 `previous_interaction_id` 很像，都是讓平台端幫你延續前一次對話脈絡。和 Claude 不同的是，Claude Messages API 通常需要每次送出完整 messages history。

| 平台 | 多輪對話常見做法 |
|---|---|
| OpenAI Responses API | `previous_response_id=response1.id` |
| Gemini Interactions API | `previous_interaction_id=interaction1.id` |
| Claude Messages API | 每次送出完整 `messages` history |

使用 `previous_response_id` 的優點是程式比較簡潔；缺點是你需要保存前一次 response id，並理解對話狀態由平台端管理。


In [ ]:
response1 = client.responses.create(
    model=MODEL,
    input="我正在學習 Python API 開發。"
)

print("第一輪：")
print(response1.output_text)
print("response1 id:", response1.id)

# TODO：使用 previous_response_id=response1.id 延續上一輪對話
# 提示：第二輪 input 可以問「根據上一句，請建議我下一步該練什麼。」
response2 = None

print()
print("第二輪：")
print("尚未完成 previous_response_id 練習")


---
## 🧩 第七節：封裝成函式

如果每次都手寫 `client.responses.create()`，程式很快會變亂。接下來把 API 呼叫封裝成函式，之後做 Streamlit App 會直接用到。

In [ ]:
def ask_ai(user_input, role="你是一位有幫助的助理", model=MODEL):
    # TODO 1：呼叫 client.responses.create()
    # TODO 2：傳入 model、instructions、input
    # TODO 3：回傳 response.output_text
    return "尚未完成 ask_ai()"


answer = ask_ai("請用 3 個重點說明 Python 函式的用途。")
print(answer)

---
## 🛡️ 第八節：基本錯誤處理

API 呼叫可能失敗，常見原因包括 API key 未設定、網路不穩、額度不足、付款設定未完成或模型名稱不可用。

In [ ]:
def ask_ai_safe(user_input, role="你是一位有幫助的助理", model=MODEL):
    try:
        # TODO 1：呼叫 client.responses.create()
        # TODO 2：成功時回傳 (True, response.output_text)
        return False, "尚未完成 ask_ai_safe()"
    except Exception as e:
        # TODO 3：失敗時回傳 (False, 錯誤訊息)
        return False, str(e)


success, result = ask_ai_safe("請用一句話說明 JSON 是什麼。")
print(result)

---
## 📊 第九節：讀取 <font color="#1a73e8"><b>usage</b></font> 並建立除錯資訊

`<font color="#1a73e8"><b>usage</b></font>` 可以幫助你追蹤 token 使用量。不同模型的計價會變動，正式估價前請查官方 pricing 頁面；本週先學會取出 <font color="#1a73e8"><b>usage</b></font>。

In [ ]:
def ask_ai_with_metadata(user_input, role="你是一位有幫助的助理", model=MODEL):
    # TODO 1：呼叫 API
    # TODO 2：回傳 dict，包含 id、model、answer、usage
    return {
        "id": None,
        "model": model,
        "answer": "尚未完成 ask_ai_with_metadata()",
        "usage": None,
    }


result = ask_ai_with_metadata("請列出學習 OpenAI API 前需要會的 3 個 Python 技能。")
print(json.dumps(result, ensure_ascii=False, indent=2, default=str))

---
## 🧪 第十節：課堂練習

請完成以下任務。這一節設計給第三堂課後半段使用，目標是把本週的 API 呼叫、函式封裝、<font color="#1a73e8"><b>usage</b></font> 讀取與 instructions 設計串起來。

### 📝 練習 A：Python 作業批改助教

1. 修改 `instructions`，讓 AI 扮演「Python 作業批改助教」
2. 輸入一段學生錯誤程式，請 AI 指出錯誤原因
3. 回答需包含「錯誤原因」、「修正方式」、「修正版程式碼」
4. 使用 `ask_ai_safe()` 或 `ask_ai_with_metadata()` 呼叫

<div style="border-left:4px solid #188038; padding:10px 12px; background:#e6f4ea; margin:10px 0;">
  <font color="#188038"><b>課堂練習</b></font><br>
  先完成練習 A，再依時間進行練習 B 與選做練習 C。
</div>


In [ ]:
student_code = '''
def add_numbers(a, b):
return a + b

print(add_numbers(3, 5))
'''

# TODO 1：設定 role，讓 AI 扮演 Python 作業批改助教
role = ""

# TODO 2：建立 prompt，放入 student_code
prompt = ""

# TODO 3：呼叫 ask_ai_safe()
success, result = False, "尚未完成課堂練習"

print(result)

---
### 💰 練習 B：<font color="#1a73e8"><b>usage</b></font> 與費用估算

第 2 週已經練過 `calculate_cost()` 的概念。本練習把它接到真實 API 回應的 `<font color="#1a73e8"><b>usage</b></font>`。

注意：模型價格會變動，正式上課前請查官方 pricing 頁面。本教材只示範「如何把 <font color="#1a73e8"><b>usage</b></font> 轉成估算流程」，不要把這裡的價格當成永久資料。

<div style="border-left:4px solid #f9ab00; padding:10px 12px; background:#fff8e1; margin:10px 0;">
  <font color="#b06000"><b>成本提醒</b></font><br>
  本節重點是學會讀取 <font color="#1a73e8"><b>usage</b></font> 與建立估算流程；實際價格請以上課當天官方 pricing 為準。
</div>


In [ ]:
def get_usage_value(usage, field_name):
    """
    從 response.usage 取值。
    usage 可能是 SDK 物件，也可能被轉成 dict，因此同時支援 getattr 與 dict.get。
    """
    if usage is None:
        return 0
    if isinstance(usage, dict):
        return usage.get(field_name, 0) or 0
    return getattr(usage, field_name, 0) or 0


def estimate_cost_from_usage(usage, model=MODEL):
    """
    根據 usage 估算費用（美元）。
    價格僅供課堂示範，實際費率請以上課當天官方 pricing 為準。
    """
    pricing = {
        "gpt-5.4-mini": {"input": 0.75 / 1_000_000, "output": 4.50 / 1_000_000},
        "gpt-5.5": {"input": 5.00 / 1_000_000, "output": 30.00 / 1_000_000},
        "gpt-4o-mini": {"input": 0.15 / 1_000_000, "output": 0.60 / 1_000_000},
    }

    # TODO 1：檢查 model 是否存在於 pricing

    # TODO 2：取出 input_tokens 與 output_tokens

    # TODO 3：計算 input_cost、output_cost、total_cost

    # TODO 4：回傳包含 model、tokens、estimated_usd、estimated_twd 的 dict
    return None, "尚未完成 estimate_cost_from_usage()"


result = ask_ai_with_metadata("請用兩句話解釋為什麼 API 呼叫需要注意成本。")
cost_info, error = estimate_cost_from_usage(result["usage"], result["model"])

print("回答：")
print(result["answer"])
print()

if error:
    print(error)
else:
    print("費用估算：")
    print(json.dumps(cost_info, ensure_ascii=False, indent=2))

---
### 🎨 練習 C（選做）：自由設計一個角色

請自訂一組 `instructions`，讓 AI 扮演與課程或個人興趣相關的角色，並觀察回答風格如何改變。

可選方向：

- 資管系學習顧問
- Python 面試官
- AI 產品經理
- 資料分析助教
- 生涯規劃教練

完成後請比較：同一個 `input` 在不同 `instructions` 下，回答內容、語氣、細節程度有什麼差異。


In [ ]:
question = "我想做一個生成式 AI 期末專題，可以從哪裡開始？"

# TODO 1：設計至少 2 個不同角色
roles = [
    "",
    "",
]

# TODO 2：用迴圈呼叫 ask_ai_safe()，觀察不同角色的回答差異
for i, role in enumerate(roles, 1):
    print(f"===== 角色 {i} =====")
    # success, result = ask_ai_safe(question, role=role)
    # print(result)
    print("尚未完成")
    print()

---
## ✅ 本週小結

| 技能 | 本週學了什麼 | 下週用在哪裡 |
|------|-------------|-------------|
| API key | Colab Secrets / 環境變數 | 所有 API 呼叫 |
| Client | `client = OpenAI()` | 建立 API 連線 |
| Responses API | `client.responses.create()` | 文字生成主線 |
| 輸入格式 | `input` 字串與 role/content list | Prompt Engineering |
| 輸出解析 | `response.output_text`、`response.id`、`response.<font color="#1a73e8"><b>usage</b></font>` | 顯示、除錯、追蹤 |
| 函式封裝 | `ask_ai()`、`ask_ai_safe()` | Streamlit App |
| 錯誤處理 | `try/except` | API 穩定性 |
| 成本觀念 | <font color="#1a73e8"><b>usage</b></font> 與費用估算 | 控制 API 花費 |

---

## 🔜 下週預告：Week 4｜Prompt Engineering 實作

下週會深入練習 system/developer instruction、清楚任務、格式限制、few-shot examples、prompt 測試與比較。
